# Loan deafult Pred

In [6]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

In [14]:
train_flag = pd.read_csv("/content/data/train/train_flag.csv")
test_flag = pd.read_csv("/content/data/test/test_flag.csv")

sample_submission = pd.read_csv("/content/data/final_submission/sample_submission.csv")

print(train_flag.shape)
print(test_flag.shape)

(261383, 3)
(46127, 2)


In [8]:
with open("/content/data/train/accounts_data_train.json") as f:
    acc_train = json.load(f)

with open("/content/data/test/accounts_data_test.json") as f:
    acc_test = json.load(f)

with open("/content/data/train/enquiry_data_train.json") as f:
    enq_train = json.load(f)

with open("/content/data/test/enquiry_data_test.json") as f:
    enq_test = json.load(f)

In [9]:
accounts_rows = []

for customer in acc_train:
    for loan in customer:
        accounts_rows.append(loan)

accounts_df = pd.DataFrame(accounts_rows)

print(accounts_df.shape)

accounts_df.head()

(1245310, 7)


,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid
0,Consumer credit,272745.000,0.0,2018-09-22,2020-02-22,0000000000000000000000100000000000000000000000...,AAA09044550
1,Consumer credit,4500.000,0.0,2018-03-08,2019-07-25,000000000000000014044000000000000000000000000000,AAA09044550
2,Credit card,80996.445,0.0,2020-06-29,NaN,000000000000000000,AAA10545297
3,Consumer credit,43771.500,0.0,2020-06-09,2020-09-09,000000000,AAA14112888
4,Credit card,10480.500,0.0,2014-09-10,NaN,0000000000000000000000000000000000000000000000...,AAA20326915


In [18]:
accounts_rows_test = []

for customer in acc_test:
    for loan in customer:
        accounts_rows_test.append(loan)

accounts_test_df = pd.DataFrame(accounts_rows_test)

print(accounts_test_df.shape)

accounts_test_df.head()

(220013, 7)


,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid
0,Consumer credit,31630.50,0.0,2014-03-30,2014-11-29,000000000000000000000000,AAA14437029
1,Consumer credit,14613.39,0.0,2014-06-01,2014-11-03,000000000000000,AAA14437029
2,Credit card,54000.00,0.0,2015-12-13,2019-09-21,0000000000000000000000000000000000000000000000...,AAA14437029
3,Consumer credit,27076.50,0.0,2015-11-11,2016-11-24,000000000000000000000000000000000000,AAA14437029
4,Credit card,225000.00,0.0,2017-07-15,2019-11-14,0000000000000000000000000000000000000000000000...,AAA14437029


In [19]:
enquiry_rows = []

for customer in enq_train:
    for enquiry in customer:
        enquiry_rows.append(enquiry)

enquiry_df = pd.DataFrame(enquiry_rows)

print(enquiry_df.shape)

enquiry_df.head()

(1909926, 4)


,enquiry_type,enquiry_amt,enquiry_date,uid
0,Interbank credit,168839,2020-11-08,AAA08065248
1,Mobile operator loan,268392,2020-09-20,AAA08065248
2,Mobile operator loan,36082,2020-06-19,AAA08065248
3,Interbank credit,180467,2019-10-22,AAA08065248
4,Cash loan (non-earmarked),227459,2020-05-24,AAA08065248


In [20]:
enquiry_rows_test = []

for customer in enq_test:
    for enquiry in customer:
        enquiry_rows_test.append(enquiry)

enquiry_test_df = pd.DataFrame(enquiry_rows_test)

print(enquiry_test_df.shape)

enquiry_test_df.head()

(337662, 4)


,enquiry_type,enquiry_amt,enquiry_date,uid
0,Car loan,143000,2020-12-13,AAA02107680
1,Real estate loan,174000,2020-12-01,AAA14437029
2,Loan for working capital replenishment,65000,2019-07-01,AAA14437029
3,Loan for working capital replenishment,118000,2020-08-05,AAA14437029
4,Car loan,12000,2020-02-28,AAA14437029


In [21]:
def parse_payment_hist(hist):

    if pd.isna(hist):
        return []

    hist = str(hist)

    return [
        int(hist[i:i+3])
        for i in range(0, len(hist), 3)
    ]

In [22]:
def payment_features(hist):

    vals = parse_payment_hist(hist)

    if len(vals) == 0:

        return pd.Series({

            "max_dpd": 0,
            "mean_dpd": 0,
            "recent_dpd": 0,
            "std_dpd": 0,

            "late_1": 0,
            "late_30": 0,
            "late_60": 0,
            "late_90": 0,

            "months_history": 0,

            "num_late": 0,

            "last_3m_late": 0,
            "last_6m_late": 0,

            "recent_mean": 0,
            "older_mean": 0,

            "dpd_trend": 0
        })

    vals = np.array(vals)

    recent_3 = vals[-3:] if len(vals) >= 3 else vals

    recent_6 = vals[-6:] if len(vals) >= 6 else vals

    older = vals[:-6] if len(vals) > 6 else vals

    recent_mean = recent_6.mean()

    older_mean = older.mean()

    trend = recent_mean - older_mean

    return pd.Series({

        "max_dpd": vals.max(),

        "mean_dpd": vals.mean(),

        "recent_dpd": vals[-1],

        "std_dpd": vals.std(),

        "late_1": np.sum(vals > 0),

        "late_30": np.sum(vals >= 30),

        "late_60": np.sum(vals >= 60),

        "late_90": np.sum(vals >= 90),

        "months_history": len(vals),

        "num_late": np.sum(vals > 0),

        "last_3m_late": np.sum(recent_3 > 0),

        "last_6m_late": np.sum(recent_6 > 0),

        "recent_mean": recent_mean,

        "older_mean": older_mean,

        "dpd_trend": trend
    })

In [23]:
payment_feats = accounts_df["payment_hist_string"].apply(payment_features)

accounts_df = pd.concat(
    [accounts_df, payment_feats],
    axis=1
)

accounts_df.head()

,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid,max_dpd,mean_dpd,recent_dpd,...,late_30,late_60,late_90,months_history,num_late,last_3m_late,last_6m_late,recent_mean,older_mean,dpd_trend
0,Consumer credit,272745.000,0.0,2018-09-22,2020-02-22,0000000000000000000000100000000000000000000000...,AAA09044550,10.0,0.588235,0.0,...,0.0,0.0,0.0,17.0,1.0,0.0,0.0,0.000000,0.909091,-0.909091
1,Consumer credit,4500.000,0.0,2018-03-08,2019-07-25,000000000000000014044000000000000000000000000000,AAA09044550,44.0,3.625000,0.0,...,1.0,0.0,0.0,16.0,2.0,0.0,0.0,0.000000,5.800000,-5.800000
2,Credit card,80996.445,0.0,2020-06-29,NaN,000000000000000000,AAA10545297,0.0,0.000000,0.0,...,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
3,Consumer credit,43771.500,0.0,2020-06-09,2020-09-09,000000000,AAA14112888,0.0,0.000000,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
4,Credit card,10480.500,0.0,2014-09-10,NaN,0000000000000000000000000000000000000000000000...,AAA20326915,917.0,190.486842,0.0,...,30.0,29.0,28.0,76.0,31.0,0.0,2.0,300.666667,181.042857,119.623810


In [24]:
payment_feats_test = accounts_test_df["payment_hist_string"].apply(payment_features)

accounts_test_df = pd.concat(
    [accounts_test_df, payment_feats_test],
    axis=1
)

accounts_test_df.head()

,credit_type,loan_amount,amount_overdue,open_date,closed_date,payment_hist_string,uid,max_dpd,mean_dpd,recent_dpd,...,late_30,late_60,late_90,months_history,num_late,last_3m_late,last_6m_late,recent_mean,older_mean,dpd_trend
0,Consumer credit,31630.50,0.0,2014-03-30,2014-11-29,000000000000000000000000,AAA14437029,0.0,0.000000,0.0,...,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.000000,0.000000
1,Consumer credit,14613.39,0.0,2014-06-01,2014-11-03,000000000000000,AAA14437029,0.0,0.000000,0.0,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.000000,0.000000
2,Credit card,54000.00,0.0,2015-12-13,2019-09-21,0000000000000000000000000000000000000000000000...,AAA14437029,0.0,0.000000,0.0,...,0.0,0.0,0.0,45.0,0.0,0.0,0.0,0.0,0.000000,0.000000
3,Consumer credit,27076.50,0.0,2015-11-11,2016-11-24,000000000000000000000000000000000000,AAA14437029,0.0,0.000000,0.0,...,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.000000,0.000000
4,Credit card,225000.00,0.0,2017-07-15,2019-11-14,0000000000000000000000000000000000000000000000...,AAA14437029,285.0,53.571429,0.0,...,9.0,8.0,7.0,28.0,10.0,2.0,5.0,187.5,17.045455,170.454545


In [25]:
for df in [accounts_df, accounts_test_df]:

    df["open_date"] = pd.to_datetime(df["open_date"])

    df["closed_date"] = pd.to_datetime(df["closed_date"])

    df["loan_duration"] = (
        df["closed_date"] - df["open_date"]
    ).dt.days

    df["loan_duration"] = df["loan_duration"].fillna(0)

    df["is_active"] = df["closed_date"].isna().astype(int)

    df["overdue_ratio"] = (
        df["amount_overdue"] /
        (df["loan_amount"] + 1)
    )

In [26]:
agg_accounts = accounts_df.groupby("uid").agg({

    "loan_amount": ["sum", "mean", "max", "std", "count"],

    "amount_overdue": ["sum", "mean", "max"],

    "overdue_ratio": ["mean", "max"],

    "loan_duration": ["mean", "max"],

    "max_dpd": ["max", "mean"],

    "mean_dpd": ["mean", "max"],

    "recent_dpd": ["max", "mean"],

    "std_dpd": ["mean"],

    "late_1": ["sum"],

    "late_30": ["sum"],

    "late_60": ["sum"],

    "late_90": ["sum"],

    "num_late": ["sum"],

    "last_3m_late": ["sum"],

    "last_6m_late": ["sum"],

    "recent_mean": ["mean"],

    "older_mean": ["mean"],

    "dpd_trend": ["mean"],

    "months_history": ["sum", "mean"],

    "is_active": ["sum"]

})

agg_accounts.columns = [
    "_".join(col).strip()
    for col in agg_accounts.columns.values
]

agg_accounts.reset_index(inplace=True)

agg_accounts.head()

,uid,loan_amount_sum,loan_amount_mean,loan_amount_max,loan_amount_std,loan_amount_count,amount_overdue_sum,amount_overdue_mean,amount_overdue_max,overdue_ratio_mean,...,late_90_sum,num_late_sum,last_3m_late_sum,last_6m_late_sum,recent_mean_mean,older_mean_mean,dpd_trend_mean,months_history_sum,months_history_mean,is_active_sum
0,AAA09044550,277245.000,138622.500,272745.000,189677.858519,2,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,0.0,0.000000,3.354545,-3.354545,33.0,16.500,0
1,AAA10545297,80996.445,80996.445,80996.445,NaN,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,6.0,6.000,1
2,AAA14112888,43771.500,43771.500,43771.500,NaN,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,3.0,3.000,0
3,AAA20326915,591597.000,73949.625,235800.000,75716.663661,8,0.0,0.0,0.0,0.0,...,28.0,31.0,0.0,2.0,37.583333,22.630357,14.952976,151.0,18.875,3
4,AAA31604840,1591960.500,318392.100,687150.000,267948.769769,5,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,78.0,15.600,3


In [27]:
agg_accounts_test = accounts_test_df.groupby("uid").agg({

    "loan_amount": ["sum", "mean", "max", "std", "count"],

    "amount_overdue": ["sum", "mean", "max"],

    "overdue_ratio": ["mean", "max"],

    "loan_duration": ["mean", "max"],

    "max_dpd": ["max", "mean"],

    "mean_dpd": ["mean", "max"],

    "recent_dpd": ["max", "mean"],

    "std_dpd": ["mean"],

    "late_1": ["sum"],

    "late_30": ["sum"],

    "late_60": ["sum"],

    "late_90": ["sum"],

    "num_late": ["sum"],

    "last_3m_late": ["sum"],

    "last_6m_late": ["sum"],

    "recent_mean": ["mean"],

    "older_mean": ["mean"],

    "dpd_trend": ["mean"],

    "months_history": ["sum", "mean"],

    "is_active": ["sum"]

})

agg_accounts_test.columns = [
    "_".join(col).strip()
    for col in agg_accounts_test.columns.values
]

agg_accounts_test.reset_index(inplace=True)

In [28]:
credit_counts = pd.crosstab(
    accounts_df["uid"],
    accounts_df["credit_type"]
).reset_index()

credit_counts_test = pd.crosstab(
    accounts_test_df["uid"],
    accounts_test_df["credit_type"]
).reset_index()

In [29]:
reference_date = pd.Timestamp("2021-01-01")

for df in [enquiry_df, enquiry_test_df]:

    df["enquiry_date"] = pd.to_datetime(df["enquiry_date"])

    df["days_since_enquiry"] = (
        reference_date - df["enquiry_date"]
    ).dt.days

In [30]:
agg_enquiry = enquiry_df.groupby("uid").agg({

    "enquiry_amt": ["count", "sum", "mean", "max", "std"],

    "days_since_enquiry": ["min", "mean"]

})

agg_enquiry.columns = [
    "_".join(col).strip()
    for col in agg_enquiry.columns.values
]

agg_enquiry.reset_index(inplace=True)

agg_enquiry.head()

,uid,enquiry_amt_count,enquiry_amt_sum,enquiry_amt_mean,enquiry_amt_max,enquiry_amt_std,days_since_enquiry_min,days_since_enquiry_mean
0,AAA08065248,11,2064658,187696.181818,364751,102098.260115,2,279.909091
1,AAA09044550,26,2659000,102269.230769,197000,50263.750511,3,289.538462
2,AAA10545297,14,1317000,94071.428571,192000,66014.525541,64,359.142857
3,AAA14112888,15,1465000,97666.666667,185000,49185.750937,180,462.400000
4,AAA20326915,1,66000,66000.000000,66000,NaN,140,140.000000


In [31]:
agg_enquiry_test = enquiry_test_df.groupby("uid").agg({

    "enquiry_amt": ["count", "sum", "mean", "max", "std"],

    "days_since_enquiry": ["min", "mean"]

})

agg_enquiry_test.columns = [
    "_".join(col).strip()
    for col in agg_enquiry_test.columns.values
]

agg_enquiry_test.reset_index(inplace=True)

In [32]:
enquiry_types = pd.crosstab(
    enquiry_df["uid"],
    enquiry_df["enquiry_type"]
).reset_index()

enquiry_types_test = pd.crosstab(
    enquiry_test_df["uid"],
    enquiry_test_df["enquiry_type"]
).reset_index()

In [33]:
train = train_flag.merge(
    agg_accounts,
    on="uid",
    how="left"
)

train = train.merge(
    credit_counts,
    on="uid",
    how="left"
)

train = train.merge(
    agg_enquiry,
    on="uid",
    how="left"
)

train = train.merge(
    enquiry_types,
    on="uid",
    how="left"
)

train.shape

(261383, 74)

In [34]:
test = test_flag.merge(
    agg_accounts_test,
    on="uid",
    how="left"
)

test = test.merge(
    credit_counts_test,
    on="uid",
    how="left"
)

test = test.merge(
    agg_enquiry_test,
    on="uid",
    how="left"
)

test = test.merge(
    enquiry_types_test,
    on="uid",
    how="left"
)

test.shape

(46127, 70)

In [35]:
train, test = train.align(
    test,
    join="left",
    axis=1,
    fill_value=0
)

In [36]:
train.fillna(0, inplace=True)

test.fillna(0, inplace=True)

In [37]:
X = train.drop(["uid", "TARGET"], axis=1)

y = train["TARGET"]

X_test = test.drop(["uid"], axis=1)

cat_features = ["NAME_CONTRACT_TYPE"]

print(X.shape)
print(X_test.shape)

(261383, 72)
(46127, 73)


In [38]:
folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))

preds = np.zeros(len(X_test))

In [39]:
for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y)):

    print("=" * 50)
    print(f"FOLD {fold+1}")
    print("=" * 50)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(

        iterations=3000,

        learning_rate=0.02,

        depth=8,

        l2_leaf_reg=5,

        loss_function="Logloss",

        eval_metric="AUC",

        random_seed=42,

        verbose=200
    )

    model.fit(

        X_train,
        y_train,

        cat_features=cat_features,

        eval_set=(X_valid, y_valid),

        early_stopping_rounds=200
    )

    oof[valid_idx] = model.predict_proba(X_valid)[:, 1]

    preds += model.predict_proba(X_test)[:, 1] / 5

FOLD 1
0:	test: 0.5878901	best: 0.5878901 (0)	total: 177ms	remaining: 8m 50s
200:	test: 0.6747663	best: 0.6747663 (200)	total: 28.4s	remaining: 6m 35s
400:	test: 0.6799680	best: 0.6799680 (400)	total: 1m	remaining: 6m 34s
600:	test: 0.6814825	best: 0.6814825 (600)	total: 1m 28s	remaining: 5m 53s
800:	test: 0.6820255	best: 0.6820734 (787)	total: 1m 57s	remaining: 5m 21s
1000:	test: 0.6822214	best: 0.6823085 (944)	total: 2m 25s	remaining: 4m 51s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6823085086
bestIteration = 944

Shrink model to first 945 iterations.
FOLD 2
0:	test: 0.5759921	best: 0.5759921 (0)	total: 221ms	remaining: 11m 1s
200:	test: 0.6665858	best: 0.6665858 (200)	total: 27.1s	remaining: 6m 17s
400:	test: 0.6725636	best: 0.6725710 (399)	total: 54.7s	remaining: 5m 54s
600:	test: 0.6755296	best: 0.6755296 (600)	total: 1m 23s	remaining: 5m 31s
800:	test: 0.6770181	best: 0.6770181 (800)	total: 1m 51s	remaining: 5m 6s
1000:	test: 0.6777156	best: 0.6777156 (

In [40]:
auc = roc_auc_score(y, oof)

print("FINAL AUC =", auc)

FINAL AUC = 0.6793667543664375


In [41]:
feature_imp = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_
})

feature_imp = feature_imp.sort_values(
    by="Importance",
    ascending=False
)

feature_imp.head(30)

,Feature,Importance
54,days_since_enquiry_mean,20.009021
32,is_active_sum,9.707261
11,loan_duration_mean,6.250295
31,months_history_mean,5.004323
30,months_history_sum,4.241079
48,enquiry_amt_count,3.607239
12,loan_duration_max,3.592570
49,enquiry_amt_sum,3.274397
3,loan_amount_max,2.942342
53,days_since_enquiry_min,2.891410


In [42]:
submission = sample_submission.copy()

submission["TARGET"] = preds

submission.head()

,uid,pred,TARGET
0,CMO22835242,0.1,0.051548
1,MRJ34316727,0.1,0.183154
2,UAV00534378,0.1,0.094176
3,IPQ08190402,0.1,0.064070
4,NQN84331006,0.1,0.075152


In [43]:
submission.to_csv(
    "final_submission_tejasv_gupta.csv",
    index=False
)

print("Submission Saved")

Submission Saved
